In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import os
import re
from scipy.interpolate import UnivariateSpline, PchipInterpolator
from sklearn.isotonic import isotonic_regression
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))
from fun_LR_hydro_memory import *

# selected_lag_type_q = 'type1_incr1'
# selected_lag_type_gw = 'type2_incr4'

lag_ranges_gw, widths_gw, incr_name_gw = make_lag_ranges(
    lag_increase=4,
    n_windows=6,
    incr_type=2
)
 
lag_ranges_q, widths_q, incr_name_q = make_lag_ranges(
    lag_increase=1,
    n_windows=6,
    incr_type=1
)
 
print('Groundwater lag ranges:\n', 
      lag_ranges_gw)
print('Streamflow lag ranges:\n', 
      lag_ranges_q)
 

Groundwater lag ranges:
 OrderedDict({'lag_0_0': (0, 0), 'lag_1_6': (1, 6), 'lag_7_17': (7, 17), 'lag_18_33': (18, 33), 'lag_34_54': (34, 54), 'lag_55_80': (55, 80)})
Streamflow lag ranges:
 OrderedDict({'lag_0_0': (0, 0), 'lag_1_2': (1, 2), 'lag_3_5': (3, 5), 'lag_6_9': (6, 9), 'lag_10_14': (10, 14), 'lag_15_20': (15, 20)})


In [2]:
       
        
def compute_memory_from_spline_variants(df_config, perc_max_r2=0.9, spline_smoothing=0.5):
    """
    Compute memory estimates from a DataFrame built for a single well (well_data).

    This minimal, robust implementation collects available (pred_memory1..6, r2_pred1..6)
    pairs across the rows in `df_config`, filters out NaNs, and fits three curve variants
    (UnivariateSpline, PCHIP, and isotonic) to estimate the predictor memory at which
    R² first reaches `perc_max_r2` of the observed maximum R².

    Returns a dict with keys: 'lag_type' (None here), 'memory_spline','memory_pchip','memory_iso',
    'x_vals','y_spline','y_pchip','y_iso'.
    """
    # Collect x (memory) and y (r2) pairs from available columns
    x_list = []
    y_list = []
    for _, row in df_config.iterrows():
        for i in range(1, 7):
            xm = row.get(f'pred_memory{i}', np.nan)
            ym = row.get(f'r2_pred{i}', np.nan)
            x_list.append(float(xm))
            y_list.append(float(ym))

    if len(x_list) == 0:
        # nothing to fit
        return {
            'lag_type': None,
            'memory_spline': np.nan,
            'memory_pchip': np.nan,
            'memory_iso': np.nan,
            'x_vals': None,
            'y_spline': None,
            'y_pchip': None,
            'y_iso': None
        }

    x = np.array(x_list)
    y = np.array(y_list)

    # Remove NaNs and sort by x
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]
    if x.size == 0:
        return {
            'lag_type': None,
            'memory_spline': np.nan,
            'memory_pchip': np.nan,
            'memory_iso': np.nan,
            'x_vals': None,
            'y_spline': None,
            'y_pchip': None,
            'y_iso': None
        }

    idx = np.argsort(x)
    x_sorted = x[idx]
    y_sorted = y[idx]
    x_dense = np.linspace(x_sorted.min(), x_sorted.max(), 500)
    r2_target = perc_max_r2 * np.nanmax(y_sorted)

    def first_cross(xvals, yvals, thr):
        above = yvals >= thr
        return xvals[np.argmax(above)] if np.any(above) else np.nan

    # Spline
    try:
        spline = UnivariateSpline(x_sorted, y_sorted, s=spline_smoothing)
        y_spline = spline(x_dense)
        mem_spline = first_cross(x_dense, y_spline, r2_target)
    except Exception:
        y_spline = None
        mem_spline = np.nan

    # PCHIP
    try:
        pchip = PchipInterpolator(x_sorted, y_sorted)
        y_pchip = pchip(x_dense)
        mem_pchip = first_cross(x_dense, y_pchip, r2_target)
    except Exception:
        y_pchip = None
        mem_pchip = np.nan

    # Isotonic
    try:
        # prefer the functional API if available
        try:
            y_iso = isotonic_regression(y_sorted)
        except Exception:
            # fallback to estimator API
            from sklearn.isotonic import IsotonicRegression
            ir = IsotonicRegression()
            # fit_transform expects X shape; use x_sorted as X to preserve monotonicity domain
            y_iso = ir.fit_transform(x_sorted, y_sorted)
        y_iso_dense = np.interp(x_dense, x_sorted, y_iso)
        mem_iso = first_cross(x_dense, y_iso_dense, r2_target)
    except Exception:
        y_iso_dense = None
        mem_iso = np.nan

    return {
        'lag_type': None,
        'memory_spline': mem_spline,
        'memory_pchip': mem_pchip,
        'memory_iso': mem_iso,
        'x_vals': x_dense,
        'y_spline': y_spline,
        'y_pchip': y_pchip,
        'y_iso': y_iso_dense,
    }


In [3]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.isotonic import IsotonicRegression

# =========================
# FUNCTION: isotonic memory
# =========================
def compute_memory_isotonic(df_config, perc_max_r2=0.9):
    # collect x (memory) and y (R2)
    x_list = []
    y_list = []

    for _, row in df_config.iterrows():
        for i in range(1, 7):
            xm = row.get(f'pred_memory{i}', np.nan)
            ym = row.get(f'r2_pred{i}', np.nan)
            x_list.append(float(xm))
            y_list.append(float(ym))

    x = np.array(x_list)
    y = np.array(y_list)

    # remove NaNs
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]

    if len(x) == 0:
        return {
            'memory_iso': np.nan,
            'x_vals': None,
            'y_iso': None
        }

    # sort by x
    idx = np.argsort(x)
    x_sorted = x[idx]
    y_sorted = y[idx]

    # isotonic fit
    ir = IsotonicRegression(increasing=True)
    y_iso = ir.fit_transform(x_sorted, y_sorted)

    # dense x for smooth plotting
    x_dense = np.linspace(x_sorted.min(), x_sorted.max(), 500)
    y_iso_dense = np.interp(x_dense, x_sorted, y_iso)

    # threshold = 90% of max R2
    r2_target = perc_max_r2 * np.nanmax(y_sorted)

    # first x where fitted curve reaches threshold
    above = y_iso_dense >= r2_target
    if np.any(above):
        memory_iso = x_dense[np.argmax(above)]
    else:
        memory_iso = np.nan

    return {
        'memory_iso': memory_iso,
        'x_vals': x_dense,
        'y_iso': y_iso_dense
    }


# =========================
# LOAD R2 RESULTS
# =========================
df_gw = pd.read_csv('../2_run_LR_gw/csv/r2_LR_rest_pos_False_trained_all.csv', dtype={"well_id": str})
df_q = pd.read_csv('../3_run_LR_Q/csv/r2_LR_rest_pos_False_trained_all.csv', dtype={"well_id": str})

# use groundwater
df = df_gw.copy()
df = df[(df['target'] == 'gw_an') & (df['predictor'] == 'pr_an')]

# =========================
# LOAD PREDICTIONS
# =========================
predictions_file = '../2_run_LR_gw/csv/gw_sim_LR_trained_all.csv'
df_pred = pd.read_csv(predictions_file, parse_dates=['date'], dtype={"well_id": str})
df_pred = df_pred[(df_pred['target'] == 'gw_an') & (df_pred['predictor'] == 'pr_an')]

# selected model for memory estimate
selected_lag_type_gw = 'type2_incr4'

# =========================
# SIMPLE MODEL NAMES
# =========================
model_name_map = {
    'type1_incr4': 'Model 1',
    'type2_incr4': 'Model 2',
    'type3_incr2': 'Model 3'
}

# =========================
# CREATE pred_memory1..6
# =========================
pattern = r"\(?\s*[-\d.]+\s*,\s*([-\d.]+)\s*\)?"

for i in range(1, 7):
    df[f'pred_memory{i}'] = np.nan

for idx in df.index:
    lr = df.at[idx, 'lag_ranges'] if 'lag_ranges' in df.columns else None

    if lr is None:
        vals = [np.nan] * 6
    else:
        matches = re.findall(pattern, str(lr))
        vals = []
        for i in range(6):
            if i < len(matches):
                try:
                    vals.append(float(matches[i]))
                except:
                    vals.append(np.nan)
            else:
                vals.append(np.nan)

    for j, v in enumerate(vals, start=1):
        df.at[idx, f'pred_memory{j}'] = v

df['pred_memory'] = df.apply(
    lambda r: np.array([r[f'pred_memory{i}'] for i in range(1, 7)]),
    axis=1
)

# =========================
# OUTPUT FOLDERS
# =========================
out_figs = 'figs_gw2'
os.makedirs(out_figs, exist_ok=True)

out_results = 'csv2'
os.makedirs(out_results, exist_ok=True)

# =========================
# COLORS
# =========================
model_colors = {
    'type1_incr4': 'blue',
    'type2_incr4': 'purple',
    'type3_incr2': 'green'
}

memory_summary = []

# =========================
# PLOT PER WELL
# =========================
for well in sorted(df['well_id'].unique()):
    well_data = df[df['well_id'] == well].copy()

    # selected model only
    df_fit = well_data[well_data['lag_type'] == selected_lag_type_gw].copy()
    max_r2_selected = df_fit['r2_all'].max() if 'r2_all' in df_fit.columns else np.nan

    if np.isnan(max_r2_selected) or max_r2_selected <= 0.4:
        print(f"Skipping well {well}: max_r2_selected={max_r2_selected}")
        continue

    fig = plt.figure(figsize=(13, 4))
    gs = gridspec.GridSpec(1, 2, width_ratios=[3, 2])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # =========================
    # LEFT: observed vs simulated
    # =========================
    pred_data = df_pred[df_pred['well_id'] == well].copy()

    for lag_type in sorted(well_data['lag_type'].unique()):
        pred_subset = pred_data[pred_data['lag_type'] == lag_type]
        color = model_colors.get(lag_type, 'grey')
        label = model_name_map.get(lag_type, lag_type)

        ax1.plot(
            pred_subset['date'],
            pred_subset['pred'],
            label=label,
            color=color,
            linewidth=1
        )

    obs = pred_data[pred_data['target'] == 'gw_an']
    if not obs.empty and 'target_val' in obs.columns:
        ax1.scatter(obs['date'], obs['target_val'], color='black', s=6, zorder=3)

    ax1.set_title(f"Well {well} – Observed vs simulated GWL, R² = {max_r2_selected:.2f}", loc='left')
    ax1.set_xlabel("Date")
    ax1.set_ylabel("GW anom (m)")
    ax1.grid(True, linestyle='--', alpha=0.5)

    handles, labels = ax1.get_legend_handles_labels()
    if labels:
        ax1.legend(fontsize=8)

    # =========================
    # RIGHT: R2 vs predictor memory
    # =========================
    for lag_type in sorted(well_data['lag_type'].unique()):
        pred_subset = well_data[well_data['lag_type'] == lag_type].copy()
        color = model_colors.get(lag_type, 'grey')
        label = model_name_map.get(lag_type, lag_type)

        xm = np.array([
            pred_subset['pred_memory1'],
            pred_subset['pred_memory2'],
            pred_subset['pred_memory3'],
            pred_subset['pred_memory4'],
            pred_subset['pred_memory5'],
            pred_subset['pred_memory6']
        ])

        ym = np.array([
            pred_subset['r2_pred1'],
            pred_subset['r2_pred2'],
            pred_subset['r2_pred3'],
            pred_subset['r2_pred4'],
            pred_subset['r2_pred5'],
            pred_subset['r2_pred6']
        ])

        ax2.plot(
            xm, ym,
            marker='o',
            linestyle='None',
            color=color,
            label=label,
            zorder=2
        )

    # isotonic fit only
    fit_results = compute_memory_isotonic(df_fit, perc_max_r2=0.9)

    if fit_results['x_vals'] is not None and fit_results['y_iso'] is not None:
        ax2.plot(
            fit_results['x_vals'],
            fit_results['y_iso'],
            '--',
            color='black',
            linewidth=1,
            label='Isotonic fit'
        )

    memory_iso = fit_results['memory_iso']

    if not np.isnan(memory_iso):
        ax2.axvline(memory_iso, color='black', linestyle='-', linewidth=1, label='GW memory')

    ax2.set_title(f"R² with increasing # of predictors\nGW memory = {memory_iso:.0f} months", loc='left')
    ax2.set_xlabel("P lags (months)")
    ax2.set_ylabel("R²")
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(fontsize=8)

    x_min = df['pred_memory'].apply(lambda a: np.nanmin(a) if hasattr(a, '__iter__') else np.nan).min()
    x_max = df['pred_memory'].apply(lambda a: np.nanmax(a) if hasattr(a, '__iter__') else np.nan).max()
    ax2.set_xlim(x_min - 5, x_max + 5)

    plt.tight_layout()
    plt.savefig(os.path.join(out_figs, f"gw_memory_{well}.png"), bbox_inches='tight')
    plt.close(fig)

    # save result
    if not np.isnan(memory_iso):
        memory_summary.append({
            'well_id': well,
            'gw_memory': memory_iso,
            'max_r2_selected_configs': max_r2_selected
        })

# # =========================
# # SAVE SUMMARY
# # =========================
if memory_summary:
    df_memory_summary = pd.DataFrame(memory_summary)
    out_csv = os.path.join(out_results, 'gw_memory.csv')

    if os.path.exists(out_csv):
        os.remove(out_csv)

    df_memory_summary.to_csv(out_csv, index=False)
else:
    print('No memory results to save')

print("\nLR models Finished! \n+Finito! \n+Terminado!\n")

Skipping well 1110003: max_r2_selected=0.201425277
Skipping well 1110004: max_r2_selected=0.136842693
Skipping well 1110005: max_r2_selected=0.126664698
Skipping well 1110006: max_r2_selected=0.134900713
Skipping well 1110007: max_r2_selected=0.162885942
Skipping well 1110008: max_r2_selected=0.360920203
Skipping well 1110009: max_r2_selected=0.270030289
Skipping well 1110010: max_r2_selected=0.248687891
Skipping well 1211010: max_r2_selected=0.223456207
Skipping well 1310024: max_r2_selected=0.241770584
Skipping well 1310025: max_r2_selected=0.131211681
Skipping well 1310026: max_r2_selected=0.368434612
Skipping well 1310027: max_r2_selected=0.34171162
Skipping well 1310033: max_r2_selected=0.295280591
Skipping well 1310036: max_r2_selected=0.239649644
Skipping well 1310064: max_r2_selected=0.233835155
Skipping well 1310069: max_r2_selected=0.334755712
Skipping well 1700013: max_r2_selected=0.173403163
Skipping well 1700019: max_r2_selected=0.181450887
Skipping well 1700023: max_r2_se

In [4]:
######## Memory computation for GW (raw vs raw  y  detrended vs detrended) ########

# NOTA: esta celda necesita que la funcion compute_memory_from_spline_variants
# (y sus imports: UnivariateSpline, PchipInterpolator) ya este definida antes.

# === Load R2 results ===
df_gw = pd.read_csv('../2_run_LR_gw/csv/r2_LR_rest_pos_False_trained_all.csv', dtype={"well_id": str})
df_q = pd.read_csv('../3_run_LR_Q/csv/r2_LR_rest_pos_False_trained_all.csv', dtype={"well_id": str})

# === Combinaciones a procesar: (target, predictor, nombre_variante) ===
combos = [
    ('gw_an',    'pr_an',    'raw'),   # raw vs raw
    ('gw_an_dt', 'pr_an_dt', 'dt'),    # detrended vs detrended
]

# Sin filtrar por target/predictor: el filtro se hace dentro del loop
df_all = df_gw.copy()

# === Load predictions (used for time series plotting) ===
predictions_file = '../2_run_LR_gw/csv/gw_sim_LR_trained_all.csv'
df_pred_all = pd.read_csv(predictions_file, parse_dates=['date'], dtype={"well_id": str})

# selected_lag_type_q = 'type1_incr1'
selected_lag_type_gw = 'type2_incr4'   # mismo tipo de incremento para raw y dt

# === Create pred_memory1..6 columns from lag_ranges ===
pattern = r"\(?\s*[-\d.]+\s*,\s*([-\d.]+)\s*\)?"
for i in range(1, 7):
    df_all[f'pred_memory{i}'] = np.nan

for idx in df_all.index:
    lr = df_all.at[idx, 'lag_ranges'] if 'lag_ranges' in df_all.columns else None
    if lr is None:
        vals = [np.nan] * 6
    else:
        s = str(lr)
        matches = re.findall(pattern, s)
        vals = []
        for i in range(6):
            if i < len(matches):
                try:
                    vals.append(float(matches[i]))
                except Exception:
                    vals.append(np.nan)
            else:
                vals.append(np.nan)
    for j, v in enumerate(vals, start=1):
        df_all.at[idx, f'pred_memory{j}'] = v

if 'pred_memory' not in df_all.columns:
    df_all['pred_memory'] = df_all.apply(
        lambda r: np.array([r[f'pred_memory{i}'] for i in range(1, 7)]), axis=1
    )

# limites del eje x del panel derecho (iguales para raw y dt)
xlim_min = df_all['pred_memory'].apply(lambda a: np.nanmin(a) if hasattr(a, '__iter__') else np.nan).min() - 5
xlim_max = df_all['pred_memory'].apply(lambda a: np.nanmax(a) if hasattr(a, '__iter__') else np.nan).max() + 5

# === Output ===
out_figs = 'figs_gw'
os.makedirs(out_figs, exist_ok=True)

out_results = 'csv'
os.makedirs(out_results, exist_ok=True)

# === Colors ===
color_palette = plt.get_cmap('tab10')
lag_types_all = sorted(df_all['lag_type'].unique()) if 'lag_type' in df_all.columns else []
specified_colors = ['blue', 'purple', 'green']
model_colors = {}
for i, name in enumerate(lag_types_all):
    if i < len(specified_colors):
        model_colors[name] = specified_colors[i]
    else:
        model_colors[name] = color_palette(i % 10)

memory_summary = []

# =====================================================================
# LOOP POR COMBINACION (raw / dt)
# =====================================================================
for target, predictor, variant in combos:

    df = df_all[(df_all['target'] == target) & (df_all['predictor'] == predictor)].copy()
    df_pred = df_pred_all[(df_pred_all['target'] == target) &
                          (df_pred_all['predictor'] == predictor)].copy()

    # === Plot per well ===
    for well in sorted(df['well_id'].unique()):
        well_data = df[df['well_id'] == well]
        df_fit = well_data[well_data['lag_type'] == selected_lag_type_gw]
        max_r2_selected = df_fit['r2_all'].max() if 'r2_all' in df_fit.columns else np.nan

        if np.isnan(max_r2_selected) or max_r2_selected <= 0.4:
            print(f"Skipping well {well} [{variant}]: max_r2_selected={max_r2_selected}")
            continue

        fig = plt.figure(figsize=(13, 4))
        gs = gridspec.GridSpec(1, 2, width_ratios=[3, 2])
        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1])

        # Left plot: observed vs predicted
        pred_data = df_pred[df_pred['well_id'] == well].copy()

        for name in sorted(well_data['lag_type'].unique()):
            mask = (pred_data['lag_type'] == name)
            pred_subset = pred_data[mask]
            color = model_colors.get(name)
            ax1.plot(
                pred_subset['date'], pred_subset['pred'],
                label=f'{name}', color=color,
                linestyle='-', linewidth=1, zorder=1,
            )

        obs = pred_data[pred_data['target'] == target]
        if not obs.empty and 'target_val' in obs.columns:
            ax1.scatter(obs['date'], obs['target_val'], color='black', s=6, zorder=2)
        ax1.set_title(
            f"Well {well} [{variant}] – Observed vs simulated GWL, R² = {np.round(max_r2_selected, 2)}",
            loc='left'
        )
        ax1.set_xlabel("Date")
        ax1.set_ylabel("GW anom (m)")
        ax1.grid(True, linestyle='--', alpha=0.5)

        handles, labels = ax1.get_legend_handles_labels()
        if labels:
            ax1.legend(fontsize=8)

        # Right plot: R² vs memory
        for name in sorted(well_data['lag_type'].unique()):
            mask = ((well_data['lag_type'] == name) &
                    (well_data['target'] == target) &
                    (well_data['predictor'] == predictor))
            pred_subset = well_data[mask]
            color = model_colors.get(name, 'red')
            label = name
            xm = np.array([pred_subset['pred_memory1'],
                           pred_subset['pred_memory2'],
                           pred_subset['pred_memory3'],
                           pred_subset['pred_memory4'],
                           pred_subset['pred_memory5'],
                           pred_subset['pred_memory6']])
            ym = np.array([pred_subset['r2_pred1'],
                           pred_subset['r2_pred2'],
                           pred_subset['r2_pred3'],
                           pred_subset['r2_pred4'],
                           pred_subset['r2_pred5'],
                           pred_subset['r2_pred6']])

            ax2.plot(xm, ym, label=label, marker='o', linestyle='None', color=color, zorder=2)

        # fitted curves + one memory line per fit
        fit_results = compute_memory_from_spline_variants(df_fit)

        fit_styles = {
            'iso':    ('y_iso',    'memory_iso',    'orange', 'Isotonic'),
            'spline': ('y_spline', 'memory_spline', 'grey',   'Spline'),
            'pchip':  ('y_pchip',  'memory_pchip',  'blue',   'PCHIP'),
        }

        mem = {}
        for key, (ycol, mcol, c, lab) in fit_styles.items():
            mem[key] = fit_results.get(mcol, np.nan)

            if fit_results['x_vals'] is not None and fit_results[ycol] is not None:
                ax2.plot(fit_results['x_vals'], fit_results[ycol],
                         '--', color=c, alpha=0.7, linewidth=.6)

            if not np.isnan(mem[key]):
                ax2.axvline(mem[key], color=c, linestyle='-', linewidth=.9,
                            label=f'{lab}: {mem[key]:.0f} mo')

        ax2.set_title(
            "R² with increasing # of predictors\n"
            f"GW memory: iso={mem['iso']:.0f}, spline={mem['spline']:.0f}, pchip={mem['pchip']:.0f} months",
            loc='left'
        )
        ax2.set_xlabel(f"{predictor} lags (months)")
        ax2.set_ylabel("R²")
        ax2.grid(True, linestyle='--', alpha=0.5)
        ax2.legend(fontsize=8)
        ax2.set_xlim(xlim_min, xlim_max)

        plt.tight_layout()
        plt.savefig(os.path.join(out_figs, f"gw_memory_{well}_{variant}.png"), bbox_inches='tight')
        plt.close(fig)

        # Store memory results if at least one is valid
        if not all(np.isnan(v) for v in mem.values()):
            memory_summary.append({
                'well_id': well,
                'variant': variant,
                'target': target,
                'predictor': predictor,
                'lag_type': selected_lag_type_gw,
                'gw_memory_iso': mem['iso'],
                'gw_memory_spline': mem['spline'],
                'gw_memory_pchip': mem['pchip'],
                'max_r2_selected_configs': max_r2_selected
            })

# =====================================================================
# write summary (fuera de los loops)
# =====================================================================
if memory_summary:
    df_memory_summary = pd.DataFrame(memory_summary)
    out_csv = os.path.join(out_results, 'gw_memory.csv')
    try:
        if os.path.exists(out_csv):
            os.remove(out_csv)
    except Exception:
        pass
    df_memory_summary.to_csv(out_csv, index=False)
else:
    print('No memory results to save')

print("\nLR models Finished! \n+Finito! \n+Terminado!\n")

Skipping well 1110003 [raw]: max_r2_selected=0.201425277
Skipping well 1110004 [raw]: max_r2_selected=0.136842693
Skipping well 1110005 [raw]: max_r2_selected=0.126664698
Skipping well 1110006 [raw]: max_r2_selected=0.134900713
Skipping well 1110007 [raw]: max_r2_selected=0.162885942
Skipping well 1110008 [raw]: max_r2_selected=0.360920203
Skipping well 1110009 [raw]: max_r2_selected=0.270030289
Skipping well 1110010 [raw]: max_r2_selected=0.248687891
Skipping well 1211010 [raw]: max_r2_selected=0.223456207
Skipping well 1310024 [raw]: max_r2_selected=0.241770584
Skipping well 1310025 [raw]: max_r2_selected=0.131211681
Skipping well 1310026 [raw]: max_r2_selected=0.368434612
Skipping well 1310027 [raw]: max_r2_selected=0.34171162
Skipping well 1310033 [raw]: max_r2_selected=0.295280591
Skipping well 1310036 [raw]: max_r2_selected=0.239649644
Skipping well 1310064 [raw]: max_r2_selected=0.233835155
Skipping well 1310069 [raw]: max_r2_selected=0.334755712
Skipping well 1700013 [raw]: max

In [5]:
######## Memory computation for STREAMFLOW (solo isotonic, sin filtro de R²) ########
# Requiere compute_memory_isotonic(df_config, perc_max_r2) definida antes

make_plots = True          # False: solo exporta el CSV (mucho más rápido)
selected_lag_type_q = 'type1_incr1'
perc_max_r2 = 0.9

# === Load R2 results ===
df_q = pd.read_csv('../3_run_LR_Q/csv/r2_LR_rest_pos_False_trained_all.csv', dtype={"gauge_id": str})
df = df_q[(df_q['target'] == 'q_an') & (df_q['predictor'] == 'pr_an')].copy()

# === Load predictions (used for time series plotting) ===
predictions_file = '../3_run_LR_Q/csv/q_sim_LR_trained_all.csv'
df_pred = pd.read_csv(predictions_file, parse_dates=['date'], dtype={"gauge_id": str})
df_pred = df_pred[(df_pred['target'] == 'q_an') & (df_pred['predictor'] == 'pr_an')]

# === Create pred_memory1..6 columns from lag_ranges ===
pattern = r"\(?\s*[-\d.]+\s*,\s*([-\d.]+)\s*\)?"
for i in range(1, 7):
    df[f'pred_memory{i}'] = np.nan

for idx in df.index:
    lr = df.at[idx, 'lag_ranges'] if 'lag_ranges' in df.columns else None
    if lr is None:
        vals = [np.nan] * 6
    else:
        matches = re.findall(pattern, str(lr))
        vals = []
        for i in range(6):
            if i < len(matches):
                try:
                    vals.append(float(matches[i]))
                except Exception:
                    vals.append(np.nan)
            else:
                vals.append(np.nan)
    for j, v in enumerate(vals, start=1):
        df.at[idx, f'pred_memory{j}'] = v

if 'pred_memory' not in df.columns:
    df['pred_memory'] = df.apply(lambda r: np.array([r[f'pred_memory{i}'] for i in range(1, 7)]), axis=1)

xlim_min = df['pred_memory'].apply(lambda a: np.nanmin(a)).min() - 5
xlim_max = df['pred_memory'].apply(lambda a: np.nanmax(a)).max() + 5

# === Output ===
out_figs = 'figs_q'
os.makedirs(out_figs, exist_ok=True)
out_results = 'csv'
os.makedirs(out_results, exist_ok=True)

# === Colors ===
color_palette = plt.get_cmap('tab10')
lag_types_all = sorted(df['lag_type'].unique())
specified_colors = ['blue', 'purple', 'green']
model_colors = {}
for i, name in enumerate(lag_types_all):
    model_colors[name] = specified_colors[i] if i < len(specified_colors) else color_palette(i % 10)

memory_summary = []

# === Loop por cuenca (sin filtro de R²) ===
for well in sorted(df['gauge_id'].unique()):
    well_data = df[df['gauge_id'] == well]
    df_fit = well_data[well_data['lag_type'] == selected_lag_type_q]
    if df_fit.empty:
        print(f"Skipping basin {well}: no rows for {selected_lag_type_q}")
        continue

    max_r2_selected = df_fit['r2_all'].max()

    # --- memoria isotonic ---
    fit_results = compute_memory_isotonic(df_fit, perc_max_r2=perc_max_r2)
    memory_iso = fit_results['memory_iso']

    memory_summary.append({
        'gauge_id': well,
        'r2': max_r2_selected,
        'q_memory_iso': memory_iso
    })

    if not make_plots:
        continue

    fig = plt.figure(figsize=(13, 4))
    gs = gridspec.GridSpec(1, 2, width_ratios=[3, 2])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # Left plot: Observed vs predicted time series
    pred_data = df_pred[df_pred['gauge_id'] == well].copy()

    for name in sorted(well_data['lag_type'].unique()):
        pred_subset = pred_data[pred_data['lag_type'] == name]
        ax1.plot(pred_subset['date'], pred_subset['pred'],
                 label=name, color=model_colors.get(name),
                 linestyle='-', linewidth=1, zorder=1)

    obs = pred_data[pred_data['target'] == 'q_an']
    if not obs.empty and 'target_val' in obs.columns:
        ax1.scatter(obs['date'], obs['target_val'], color='black', s=6, zorder=2)
    ax1.set_title(f"Basin {well} – observed vs simulated Q, R² = {np.round(max_r2_selected, 2)}", loc='left')
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Q anom (mm)")
    ax1.grid(True, linestyle='--', alpha=0.5)
    handles, labels = ax1.get_legend_handles_labels()
    if labels:
        ax1.legend(fontsize=8)

    # Right plot: R² vs memory
    for name in sorted(well_data['lag_type'].unique()):
        pred_subset = well_data[well_data['lag_type'] == name]
        xm = np.array([pred_subset[f'pred_memory{i}'] for i in range(1, 7)])
        ym = np.array([pred_subset[f'r2_pred{i}'] for i in range(1, 7)])
        ax2.plot(xm, ym, label=name, marker='o', linestyle='None',
                 color=model_colors.get(name, 'red'), zorder=2)

    # isotonic fit only
    if fit_results['x_vals'] is not None and fit_results['y_iso'] is not None:
        ax2.plot(fit_results['x_vals'], fit_results['y_iso'],
                 '--', color='black', linewidth=1, label='Isotonic fit')
    if not np.isnan(memory_iso):
        ax2.axvline(memory_iso, color='black', linestyle='-', linewidth=1,
                    label=f'Q memory = {memory_iso:.0f} mo')

    ax2.set_title(f"R² with increasing # of predictors\nQ memory = {memory_iso:.0f} months", loc='left')
    ax2.set_xlabel("Pr_an lags (months)")
    ax2.set_ylabel("R²")
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(fontsize=8)
    ax2.set_xlim(xlim_min, xlim_max)

    plt.tight_layout()
    plt.savefig(os.path.join(out_figs, f"q_memory_{well}.png"), bbox_inches='tight')
    plt.close(fig)

# === Write summary ===
if memory_summary:
    df_memory_summary = pd.DataFrame(memory_summary)
    out_csv = os.path.join(out_results, 'q_memory.csv')
    try:
        if os.path.exists(out_csv):
            os.remove(out_csv)
    except Exception:
        pass
    df_memory_summary.to_csv(out_csv, index=False)
    print(f"Saved {len(df_memory_summary)} basins to {out_csv}")
else:
    print('No memory results to save')

print("\nLR models Finished! \nFinito! \nTerminado!\n")

Saved 484 basins to csv\q_memory.csv

LR models Finished! 
Finito! 
Terminado!

